In [ ]:
# PREAMBLE will be replaced
dcg_data = {
    'test_data': 'training_data.txt',
    'model_directories': [
        'l1_subset',
        'l1_full',
    ],
    'indices': None  # if None, will use all all data in test_data, else will subset into test data
}

In [ ]:
import yaml
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('white')

from dcg.recorder import SavedModel
from dcg.cell_loader import CellLoader
from dcg.configuration import DEVICE

np.random.seed(100)
torch.manual_seed(100)

def mask(target, graph, model, ind=0):
    msk = model.output_mask(torch.unsqueeze(graph, 0))
    return target[ind, ...][msk[0, ind]]

# mse of single graph
def MSE(target, prediction, graph, model):
    msk = model.output_mask(torch.unsqueeze(graph, 0)).numpy()
    return np.ma.masked_array(np.square(target - prediction), 
                              mask=~msk[0]
                             ).mean(axis=(1, 2)).data

# get min, max, median indices
def representative_indices(mses):
    metric = mses.sum(axis=1)
    median = np.argmin(np.abs(np.median(metric) - metric))
    return [np.argmin(metric), median, np.argmax(metric)]

In [ ]:
data = {os.path.basename(d): 
        {'model': SavedModel(d, need_metrics=True)} 
        for d in dcg_data['model_directories']}
first_model = data[os.path.basename(dcg_data['model_directories'][0])]

In [ ]:
# use first model config to build loader
loader = first_model['model'].config.build_loader()

if dcg_data['indices'] is not None:
    loader.indices = dcg_data['indices']

cells = loader.load_cells(dcg_data['test_data'], apply_norm=False, only_test=True)
print(f'Loaded {cells} graphs from "{dcg_data["model_directories"][0]}/config.yaml"')

In [ ]:
print('Mean MSEs:')
mses = pd.DataFrame(
    columns=first_model['model'].config.target_features,
    index=data.keys()
)
for k, v in data.items():
    mod = v['model']
    v['mses'] = np.zeros((len(loader.data['test'].dataset), mod.model.num_targets))
    v['actual_total'] = np.zeros((len(loader.data['test'].dataset), mod.model.num_targets))
    v['predict_total'] = np.zeros((len(loader.data['test'].dataset), mod.model.num_targets))

    ind = 0
    for g, t in loader.test:
        te = mod.model.predict(g.to(DEVICE)).cpu()
        for i in range(g.shape[0]):  # each graph in batch
            v['mses'][ind+i] = MSE(t[i], te[i], g[i], mod.model)
            v['actual_total'][ind+i, :] = (t[i, :]*g[i, 0]).sum(axis=(1,2))
            v['predict_total'][ind+i, :] = (te[i, :]*g[i, 0]).sum(axis=(1,2))
        ind += g.shape[0]
    mses.loc[k, :] = v['mses'].mean(axis=0)
mses

In [ ]:
values = pd.concat(
    [pd.DataFrame(dat['mses'], columns=loader.target_features).assign(Model=name)
     for name, dat in data.items()
    ]
)
f, axs = plt.subplots(len(loader.target_features), 1, 
                      figsize=(10, 4*(len(loader.target_features)+1)), 
                      squeeze=False)
for ax, name in zip(axs, loader.target_features):
    ax=ax[0]
    sns.histplot(data=values, x=name, hue='Model', 
                 log_scale=True, kde=True, 
                 element='step', ax=ax)
    ax.set_title(name)
    ax.set_xlabel('MSE')

In [ ]:
for name, value in data.items():
    mod = value['model']
    inds = representative_indices(value['mses'])
    ind_names = ['Min', 'Median', 'Max']
    graphs, targets = loader.data['test'].dataset[inds]
    # use list comprehension for handling object arrays
    graphs = [torch.from_numpy(g).to(torch.float) for g in graphs]
    targets = [torch.from_numpy(t).to(torch.float) for t in targets]
    prediction = [mod.model.predict(g.unsqueeze(0).to(DEVICE)).cpu().squeeze(0)
                  for g in graphs]
    
    rows = 3
    cols = mod.model.num_targets

    f, axs = plt.subplots(rows, cols, figsize=(6*cols, 4*rows), sharex=True, squeeze=False)
    for col in range(cols):
        for row in range(rows):
            values = pd.concat(
                [pd.DataFrame(mask(targets[row], graphs[row], mod.model, col),
                              columns=["value"]).assign(type='Actual'),
                 pd.DataFrame(mask(prediction[row], graphs[row], mod.model, col),
                              columns=["value"]).assign(type='Predicted'),
                ])
            sns.histplot(data=values, x='value', hue='type', kde=True, ax=axs[row, col])
            axs[row, col].set_xlabel('')
            if col != cols-1 or row != rows-1:
                axs[row, col].get_legend().remove()
            else:
                axs[row, col].get_legend().set_title('')
    
    f.suptitle(name)
    
    for ax, name in zip(axs[0, :], loader.target_features):
        ax.set_title(name)
        
    for ax, name, ind in zip(axs[:, 0], ind_names, inds):
        ax.set_ylabel(f'{name}: mean(MSE) = {value["mses"][ind, :].mean():.5f}')
        
    plt.show()

In [ ]:
for name, value in data.items():
    mod = value['model']
    inds = representative_indices(value['mses'])
    ind_names = ['Min', 'Median', 'Max']
    graphs, targets = loader.data['test'].dataset[inds]
    # use list comprehension for handling object arrays
    graphs = [torch.from_numpy(g).to(torch.float) for g in graphs]
    targets = [torch.from_numpy(t).to(torch.float) for t in targets]
    prediction = [mod.model.predict(g.unsqueeze(0).to(DEVICE)).cpu().squeeze(0)
                  for g in graphs]
    
    rows = 3
    cols = mod.model.num_targets

    f, axs = plt.subplots(rows, cols, figsize=(6*cols, 4*rows), sharex=True, squeeze=False)
    for col in range(cols):
        for row in range(rows):
            ax = axs[row, col]
            sns.scatterplot(x=mask(targets[row], graphs[row], mod.model, col), 
                            y=mask(prediction[row], graphs[row], mod.model, col), 
                            ax=ax,
                            alpha=0.1
                           )
            lims = [
                np.min([ax.get_xlim(), ax.get_ylim()]),
                np.max([ax.get_xlim(), ax.get_ylim()]),
            ]
            ax.plot(lims, lims, 'k-', zorder=0) ;
            axs[row, col].set_xlabel('')
            
    for col in range(cols):
        axs[row, col].set_xlabel('Actual')
        
    f.suptitle(name)
        
    for ax, name, ind in zip(axs[:, 0], ind_names, inds):
        ax.set_ylabel(f'{name}: mean(MSE) = {value["mses"][ind, :].mean():.5f}')
    
    for ax, name in zip(axs[0, :], loader.target_features):
        ax.set_title(name)
        
    plt.show()

In [ ]:
for name, value in data.items():
    rows = 1
    cols = mod.model.num_targets
    f, axs = plt.subplots(rows, cols, figsize=(6*cols, 4*rows), squeeze=False)
    for col in range(cols):
        for row in range(rows):
            ax = axs[row, col]
            sns.scatterplot(x=value['actual_total'][:, col], y=value['predict_total'][:, col], ax=ax)
            ax.set_xlabel('True Total')
            ax.set_ylabel('Predicted Total')
            lims = [
                np.min([ax.get_xlim(), ax.get_ylim()]),
                np.max([ax.get_xlim(), ax.get_ylim()]),
            ]
            ax.set_title(name)
            ax.plot(lims, lims, 'k-', zorder=0)
    plt.show()

In [ ]:
for name, dat in data.items():
    mod = dat['model']
    ylim = mod.metrics[['train_loss', 'val_loss']].quantile(.9).max()
    sns.lineplot(x='epochs', y='train_loss', data=mod.metrics, label='Training')
    ax = sns.lineplot(x='epochs', y='val_loss', data=mod.metrics, label='Validation')
    ax.set_ylim(top=ylim, bottom=0)
    ypos = mod.metrics.loc[mod.epoch, ['train_loss', 'val_loss']].max()
    end_pos = ax.transData.inverted().transform(ax.transData.transform([mod.epoch, ypos]) + [0, 20])
    ax.annotate("", xy=(mod.epoch, ypos), xytext=end_pos, arrowprops=dict(arrowstyle='->', color='k'), label='Chosen Epoch')
    ax.legend()
    ax.set_xlabel('Epochs')
    ax.set_ylabel('Loss')
    plt.title(name)
    plt.show()